<a href="https://colab.research.google.com/github/jotafortunat/atvdLinguagem3/blob/main/atividade_pratica_aula4_kpis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Atividade Prática — Aula 4: Análise de Dados e Construção de Indicadores (KPIs)

Esta atividade foi construída com base na Aula 4, que marca a transição **do dado limpo para a tomada de decisão estratégica**, com foco em **KPIs, métricas derivadas, dimensões de análise, rankings e insights executivos**. fileciteturn5file0

## Ideia central da aula
Antes do código, vem a **pergunta de negócio**.  
Depois, definimos:
1. a **métrica**
2. a **dimensão**
3. a **agregação**
4. e só então construímos a resposta em tabela e em texto. fileciteturn5file0

## Regras da atividade
- O notebook **orienta**, mas você deve **escrever os códigos**.
- Sempre que gerar uma tabela agregada, escreva **1 ou 2 frases interpretando o resultado**.
- O objetivo não é apenas calcular números, mas comunicar decisões.

## Dataset da atividade
Arquivo: `vendas_brasil_aula4_kpis.csv`


## 1. Preparação do ambiente

Importe as bibliotecas necessárias para a atividade.

**Sugestão:**  
- `pandas`
- `numpy`

Se desejar, você também pode usar uma biblioteca de visualização mais adiante, mas o foco desta aula é **tabela agregada + insight**.


In [2]:
import pandas as pd
import numpy as np

# Configuração para evitar notação científica no pandas e facilitar a leitura de moeda
pd.options.display.float_format = '{:,.2f}'.format

## 2. Leitura da base

Leia o arquivo `vendas_brasil_aula4_kpis.csv` em um DataFrame chamado `df`.

Depois:
1. exiba as primeiras linhas
2. veja o tamanho da base
3. inspecione os tipos das colunas
4. confirme se `data_venda` está pronta para análises temporais


In [3]:
# Lendo o CSV e exibindo as informações essenciais
df = pd.read_csv('vendas_brasil_aula4_kpis.csv')

print("Tamanho da base:", df.shape)
display(df.head())
print("\nTipos de dados:")
print(df.dtypes)

Tamanho da base: (420, 10)


,pedido_id,data_venda,uf,canal_venda,categoria,produto,quantidade_pedidos,preco_unitario,receita,lucro
0,5000,2025-02-14,RJ,Marketplace,Periféricos,Mouse Gamer,2,237.65,475.30,239.08
1,5001,2025-02-17,PR,Online,Informática,Monitor 27,4,"1,431.79","5,727.16","1,769.48"
2,5002,2025-07-22,RJ,Marketplace,Informática,Monitor 27,5,"1,446.22","7,231.10","2,449.85"
3,5003,2025-12-08,SP,Marketplace,Periféricos,Teclado Mecânico,7,399.84,"2,798.88","1,356.67"
4,5004,2025-01-04,GO,Marketplace,Telefonia,Smartphone X,4,"3,134.85","12,539.40","3,656.84"



Tipos de dados:
pedido_id               int64
data_venda             object
uf                     object
canal_venda            object
categoria              object
produto                object
quantidade_pedidos      int64
preco_unitario        float64
receita               float64
lucro                 float64
dtype: object


## 3. Perguntas antes do código

A aula enfatiza que a maior armadilha é começar com `groupby()` sem saber o que se quer responder. fileciteturn5file0

### Tarefa
Antes de programar, escreva em markdown respostas curtas para estas perguntas:

1. Qual canal de venda é o mais eficiente?
- Vamos descobrir agregando receita e lucro por canal_venda e calculando a margem para ver quem traz mais dinheiro limpo.
2. Qual categoria parece mais rentável?
- Analisando a métrica derivada de margem de lucro (lucro/receita) agrupada pela dimensão categoria.
3. Quais produtos são os campeões por lucro?
- Criando um ranking (Top N) agrupado por produto e ordenado pela soma de lucro.
4. O negócio está crescendo ao longo do tempo?
- Agrupando receita pela dimensão temporal data_venda (convertida para mês).
5. Quais KPIs globais resumem melhor a operação?
- Receita Total, Lucro Total, Ticket Médio e Margem de Lucro percentual.
Depois disso, avance para o código.


## 4. KPIs globais da operação

Com base nas “4 Métricas de Ouro do Varejo” apresentadas na aula, calcule os seguintes KPIs globais da base inteira: fileciteturn5file0

- **Receita Total**
- **Lucro Total**
- **Margem de Lucro (%)**
- **Ticket Médio**

### Orientação
- Pense primeiro na fórmula de cada KPI
- Escreva os cálculos em células separadas ou em uma pequena tabela-resumo
- Se quiser, formate os resultados para facilitar a leitura


In [4]:
receita_total = df['receita'].sum()
lucro_total = df['lucro'].sum()
quantidade_itens = df['quantidade_pedidos'].sum()

margem_global = (lucro_total / receita_total) * 100
ticket_medio = receita_total / quantidade_itens

# Criando um dicionário/tabela resumo
kpis_globais = pd.DataFrame({
    'Receita Total (R$)': [receita_total],
    'Lucro Total (R$)': [lucro_total],
    'Margem de Lucro (%)': [margem_global],
    'Ticket Médio (R$)': [ticket_medio]
})

display(kpis_globais)

,Receita Total (R$),Lucro Total (R$),Margem de Lucro (%),Ticket Médio (R$)
0,"1,921,492.22","544,496.15",28.34,"1,382.37"


### Interpretação
Escreva 3 a 5 linhas respondendo:
- O negócio parece grande ou pequeno?
- O lucro acompanha bem a receita?
- O ticket médio sugere compras de maior valor ou de menor valor?


## 5. Métricas derivadas

A aula mostra que **somar valores não basta**.  
Precisamos criar métricas derivadas para avaliar eficiência e comportamento. fileciteturn5file0

### Tarefa
Crie, quando fizer sentido:
- uma coluna de **margem_lucro** (`lucro / receita`)
- uma lógica para **ticket_medio** da operação ou por grupo

### Reflexão
Explique:
1. Por que receita alta não significa automaticamente melhor desempenho?
2. Em que situação a margem ajuda mais do que a receita bruta?


In [5]:
# Criando as métricas diretamente no DataFrame
df['margem_lucro_perc'] = (df['lucro'] / df['receita']) * 100
df['ticket_medio'] = df['receita'] / df['quantidade_pedidos']

display(df[['pedido_id', 'receita', 'lucro', 'margem_lucro_perc', 'ticket_medio']].head())

,pedido_id,receita,lucro,margem_lucro_perc,ticket_medio
0,5000,475.30,239.08,50.30,237.65
1,5001,"5,727.16","1,769.48",30.90,"1,431.79"
2,5002,"7,231.10","2,449.85",33.88,"1,446.22"
3,5003,"2,798.88","1,356.67",48.47,399.84
4,5004,"12,539.40","3,656.84",29.16,"3,134.85"


## 6. Fatiando a realidade: análise por dimensão

A aula mostra que um KPI isolado não decide nada.  
É preciso perguntar: **como isso varia por canal, categoria, região e tempo?** fileciteturn5file0

### Tarefa
Liste, em markdown, quais colunas do dataset representam:
- métrica
- dimensão
- tempo


## 7. Análise por canal

### Pergunta de negócio
**Qual canal de venda é o mais eficiente?**

### Tarefa
Crie uma tabela agregada por `canal_venda` contendo, no mínimo:
- Receita Total
- Lucro Total
- Margem Média ou Margem Calculada
- Quantidade de pedidos
- Ticket Médio

### Depois responda
1. Qual canal gera mais receita?
2. Qual canal gera maior eficiência?
3. Se você fosse gestor, priorizaria volume ou margem?


In [6]:
# Tabela agregada por canal de venda
canal = df.groupby('canal_venda').agg(
    Receita_Total=('receita', 'sum'),
    Lucro_Total=('lucro', 'sum'),
    Pedidos=('quantidade_pedidos', 'sum')
).reset_index()

# Métricas derivadas da agregação
canal['Margem (%)'] = (canal['Lucro_Total'] / canal['Receita_Total']) * 100
canal['Ticket_Medio'] = canal['Receita_Total'] / canal['Pedidos']

canal = canal.sort_values('Receita_Total', ascending=False)
display(canal)

,canal_venda,Receita_Total,Lucro_Total,Pedidos,Margem (%),Ticket_Medio
1,Marketplace,"657,802.09","185,802.57",493,28.25,"1,334.28"
0,Loja Física,"640,797.06","174,583.16",445,27.24,"1,439.99"
2,Online,"622,893.07","184,110.42",452,29.56,"1,378.08"


### Insight obrigatório
Escreva 1 ou 2 frases interpretando sua tabela, no estilo executivo:
- não apenas “o Online foi maior”
- mas o que isso **significa para a decisão**


## 8. Análise por categoria

### Pergunta de negócio
**Qual categoria é mais lucrativa e qual parece mais eficiente?**

### Tarefa
Crie uma tabela agregada por `categoria` com:
- Receita Total
- Lucro Total
- Margem
- Ticket Médio

### Depois responda
1. A categoria de maior receita também é a de maior margem?
2. Existe alguma categoria que parece vender bem, mas ser pouco eficiente?


In [7]:
# Tabela agregada por categoria
cat = df.groupby('categoria').agg(
    Receita=('receita', 'sum'),
    Lucro=('lucro', 'sum')
).reset_index()

cat['Margem (%)'] = (cat['Lucro'] / cat['Receita']) * 100
cat = cat.sort_values('Receita', ascending=False)

display(cat)

,categoria,Receita,Lucro,Margem (%)
2,Telefonia,"866,292.45","219,975.58",25.39
0,Informática,"805,376.44","219,090.15",27.20
3,Áudio,"132,900.25","55,006.02",41.39
1,Periféricos,"116,923.08","50,424.40",43.13


## 9. Análise por região (UF)

### Pergunta de negócio
**Quais UFs concentram a operação?**

### Tarefa
Monte uma tabela por `uf` com:
- Receita Total
- Lucro Total
- Participação percentual da receita
- Ticket Médio

### Depois responda
1. Quais UFs concentram maior receita?
2. Existe concentração regional?
3. Se fosse necessário expandir ou reforçar logística, por onde você começaria?


In [8]:
# Agregação por UF
uf = df.groupby('uf').agg(
    Receita=('receita', 'sum'),
    Lucro=('lucro', 'sum')
).reset_index()

# Participação percentual
uf['Participacao_Receita (%)'] = (uf['Receita'] / receita_total) * 100
uf = uf.sort_values('Receita', ascending=False)

display(uf.head(5))

,uf,Receita,Lucro,Participacao_Receita (%)
9,SP,"246,835.50","71,022.16",12.85
6,RJ,"218,631.91","63,960.39",11.38
4,PE,"208,457.44","58,183.95",10.85
3,MG,"207,614.69","61,122.25",10.80
8,SC,"198,143.47","60,272.32",10.31


## 10. O motor da análise: agregações (groupby)

Nos slides, a lógica central é:
- `receita -> sum`
- `margem -> mean` ou cálculo derivado
- `produto -> count`
fileciteturn5file0

### Tarefa
Escolha **duas dimensões diferentes** e construa tabelas agregadas mostrando que você entendeu essa lógica.

Exemplos:
- por canal
- por categoria
- por UF
- por mês


## 11. Campeões e vilões (Top N)

A aula destaca o poder dos rankings curtos e focados. fileciteturn5file0

### Tarefa
Crie:
- Top 10 produtos por **lucro**
- Top 10 produtos por **receita**
- 5 categorias ou UFs com **pior margem** ou pior desempenho

### Perguntas
1. Quem são os “campeões”?
2. Quem são os “vilões”?
3. Que ação tática poderia ser tomada com base nisso?


In [11]:
# Campeões por Lucro
top10_produtos = df.groupby('produto')['lucro'].sum().reset_index().sort_values('lucro', ascending=False).head(10)

# Vilões por Margem (Categorias com menor rentabilidade)
viloes_margem = cat.sort_values('Margem (%)', ascending=True).head(5)

print("--- TOP PRODUTOS POR LUCRO ---")
display(top10_produtos)
print("\n--- CATEGORIAS COM PIOR MARGEM ---")
display(viloes_margem)

--- TOP PRODUTOS POR LUCRO ---


,produto,lucro
4,Notebook Pro,"153,094.84"
5,Smartphone X,"118,198.25"
6,Tablet Plus,"101,777.33"
2,Monitor 27,"65,995.31"
7,Teclado Mecânico,"33,939.07"
0,Caixa de Som,"28,316.37"
1,Headset Pro,"26,689.65"
3,Mouse Gamer,"16,485.33"



--- CATEGORIAS COM PIOR MARGEM ---


,categoria,Receita,Lucro,Margem (%)
2,Telefonia,"866,292.45","219,975.58",25.39
0,Informática,"805,376.44","219,090.15",27.20
3,Áudio,"132,900.25","55,006.02",41.39
1,Periféricos,"116,923.08","50,424.40",43.13


## 12. Análise temporal

A aula reforça que sem tempo a análise fica estática. fileciteturn5file0

### Tarefa
1. Converta `data_venda` para data, se necessário
2. Crie colunas auxiliares como:
   - mês
   - trimestre (opcional)
3. Gere uma tabela agregada por mês com:
   - receita
   - lucro
   - ticket médio ou margem

### Depois responda
1. O lucro parece crescer ao longo do tempo?
2. O último trimestre foi melhor que o anterior?
3. Quais meses concentram as vendas de fim de ano?


In [12]:
# Convertendo data e extraindo mês (Período)
df['data_venda'] = pd.to_datetime(df['data_venda'])
df['mes'] = df['data_venda'].dt.to_period('M').astype(str)

# Agregação mensal
mes = df.groupby('mes').agg(
    Receita=('receita', 'sum'),
    Lucro=('lucro', 'sum')
).reset_index().sort_values('mes')

display(mes)

,mes,Receita,Lucro
0,2025-01,"161,571.89","46,515.14"
1,2025-02,"108,587.03","29,998.87"
2,2025-03,"87,902.22","23,923.66"
3,2025-04,"148,930.46","43,094.00"
4,2025-05,"106,355.17","30,334.32"
5,2025-06,"134,685.83","38,592.70"
6,2025-07,"196,651.96","57,913.44"
7,2025-08,"160,240.62","45,881.36"
8,2025-09,"109,291.03","31,938.44"
9,2025-10,"158,226.19","41,463.02"


## 13. Fluxo de investigação do analista

A aula apresenta a sequência:
**Pergunta -> Colunas necessárias -> Tipo de resposta/gráfico**. fileciteturn5file0

### Tarefa
Preencha, em markdown, pelo menos 3 exemplos no formato:

- **Pergunta de negócio:** ...
- **Colunas necessárias:** ...
- **Métrica/KPI:** ...
- **Tipo de resposta esperada:** tabela, ranking, tendência etc.


## 14. O insight exige palavras

Segundo a aula, o mercado valoriza quem comunica decisões, não apenas quem roda código. fileciteturn5file0

### Tarefa
Escolha **duas tabelas agregadas** que você criou e escreva, para cada uma, **1 ou 2 frases de interpretação executiva**.

Exemplo de estilo:
- “O canal X concentra maior volume de receita, mas o canal Y apresenta margem superior.”
- “Recomenda-se revisar custos/expansão/priorização com base nesse resultado.”


## 15. Case Varejo Brasil — Entrega final

A aula propõe quatro entregas centrais para o gestor. fileciteturn5file0

### Sua missão final
Organize o notebook para entregar:

1. **KPIs Globais**
2. **Tabelas por Dimensões Críticas**
   - por canal
   - por categoria
3. **Top 10 Produtos por Lucro**
4. **Conclusão final com pelo menos uma decisão tática**

### Importante
A conclusão final deve ser escrita em linguagem clara, direta e orientada à decisão.


## 16. Desafio extra (opcional)

Monte uma pequena tabela executiva final com:
- Receita Total
- Lucro Total
- Margem
- Ticket Médio

e depois crie uma segunda tabela por canal com os mesmos indicadores.

Compare os resultados e explique:
- o KPI global esconde alguma diferença importante entre os canais?


In [ ]:
# Desafio extra opcional


## 17. Entrega esperada

Seu notebook deve demonstrar:
- organização
- fórmulas corretas de KPI
- uso adequado de `groupby`, `agg`, `sort_values`
- leitura temporal coerente
- interpretação escrita dos resultados

### Mensagem principal da aula
Transformar linhas de dados limpos em **apoio real para decisões de negócio**. fileciteturn5file0
